# Filtrant — CV Screening ML Pipeline

This notebook covers the full ML cycle:
1. Data loading and exploration
2. Feature engineering & analysis
3. Model comparison
4. Full evaluation (AUC, confusion matrix, feature importance)
5. Export of the final model → `model.joblib`

**To retrain**: export candidates from the API (`GET /api/v1/candidates/export.csv`), label the `recommendation` column, then re-run all cells.

## 0 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report,confusion_matrix,roc_auc_score,roc_curve,ConfusionMatrixDisplay,)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42


FEATURE_COLUMNS = [
    "total_years_experience",
    "num_positions",
    "avg_tenure_months",
    "education_level_score",
    "total_skills_count",
    "has_certifications",
    "language_count",
    "section_completeness_score",
    "max_language_score",
    "has_senior_title",
    "career_gap_months",
    "latest_job_duration",
    "has_summary",
    "num_certifications",
    "parse_quality_score",
]

MODEL_PATH = "model.joblib"

print(f"Setup OK — {len(FEATURE_COLUMNS)} features")

## 1 · Data Loading

Change `DATA_PATH` according to your source:
- `synthetic_train.csv` → synthetic data (500 rows, to get started)
- `candidates_export.csv` → real data exported from the API

In [ ]:
DATA_PATH = "candidates_export.csv" 

df = pd.read_csv(DATA_PATH )

# Quick validation
missing_cols = [c for c in FEATURE_COLUMNS + ["recommendation"] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in CSV: {missing_cols}")

# Keep only Invite / Reject (ignore 'pending')
df = df[df["recommendation"].isin(["Invite", "Reject"])].copy()
df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].fillna(0)

print(f"Dataset loaded: {len(df)} rows")
print(f"  Invite : {(df['recommendation'] == 'Invite').sum()}")
print(f"  Reject : {(df['recommendation'] == 'Reject').sum()}")

if len(df) < 50:
    print("\n Fewer than 50 rows — results will not be reliable.")
    print("   Export more labelled CVs from the dashboard before retraining.")

df.head()

## 2 · Data Exploration (EDA)

In [ ]:
# ── Descriptive statistics by class ─────────────────────────────────────────
df.groupby("recommendation")[FEATURE_COLUMNS].mean().T.style.background_gradient(axis=1, cmap="RdYlGn")

In [ ]:
# ── Distribution of each feature by class ───────────────────────────────────
n = len(FEATURE_COLUMNS)
ncols = 4
nrows = (n + ncols - 1) // ncols  # ceil division

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    for label, color in [("Invite", "#4CAF50"), ("Reject", "#F44336")]:
        axes[i].hist(
            df.loc[df["recommendation"] == label, col],
            bins=20, alpha=0.6, color=color, label=label, density=True
        )
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature distribution by class", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation between features ─────────────────────────────────────────────
plt.figure(figsize=(9, 7))
corr = df[FEATURE_COLUMNS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, linewidths=0.5)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

## 3 · Train / Test Split

In [ ]:
X = df[FEATURE_COLUMNS].values
y = (df["recommendation"] == "Invite").astype(int).values  # 0=Reject, 1=Invite

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train)} rows")
print(f"Test:  {len(X_test)} rows")
print(f"Invite ratio (train): {y_train.mean():.1%}")
print(f"Invite ratio (test):  {y_test.mean():.1%}")

## 4 · Model Comparison

In [ ]:
CANDIDATES = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "GradientBoosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, pipeline in CANDIDATES.items():
    cv_f1 = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1").mean()
    pipeline.fit(X_train, y_train)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    results.append({"Model": name, "CV F1 (train)": cv_f1, "AUC (test)": test_auc})
    print(f"{name:<22}  CV F1={cv_f1:.3f}  AUC={test_auc:.3f}")

results_df = pd.DataFrame(results).set_index("Model")
best_name = results_df["AUC (test)"].idxmax()
print(f"\n→ Best model: {best_name} (AUC={results_df.loc[best_name, 'AUC (test)']:.3f})")

In [ ]:
# ── Comparison chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
results_df.plot(kind="bar", ax=ax, color=["#5C9BD6", "#F4A261"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model comparison")
ax.set_xticklabels(results_df.index, rotation=0)
ax.legend(loc="lower right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

## 5 · Detailed Evaluation of the Best Model

In [ ]:
best_pipeline = CANDIDATES[best_name]
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(f"=== {best_name} — classification report ===")
print(classification_report(y_test, y_pred, target_names=["Reject", "Invite"]))

In [ ]:
# ── Confusion matrix + ROC Curve ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["Reject", "Invite"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion matrix")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#4CAF50", lw=2, label=f"AUC = {auc_score:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.suptitle(f"{best_name}", fontsize=13)
plt.tight_layout()
plt.show()

## 6 · Feature Importance

In [ ]:
clf = best_pipeline.named_steps["clf"]

if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
    title = "Feature importance (gain)"
elif hasattr(clf, "coef_"):
    importances = np.abs(clf.coef_[0])
    title = "Feature importance (|coefficient| LogReg)"
else:
    importances = np.ones(len(FEATURE_COLUMNS))
    title = "Feature importance not available"

fi_df = pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
fi_df = fi_df.sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(fi_df["feature"], fi_df["importance"], color="#5C9BD6")
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xlabel("Importance")
ax.set_title(title)
plt.tight_layout()
plt.show()

## 7 · Confidence Score Distribution Analysis

In [ ]:
# Verify the model is well calibrated: confidence scores should be
# spread out, not all clustered near 0 or 1.
probas_all = best_pipeline.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(probas_all[y_test == 1], bins=20, alpha=0.7, color="#4CAF50", label="Invite (actual)")
ax.hist(probas_all[y_test == 0], bins=20, alpha=0.7, color="#F44336", label="Reject (actual)")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Threshold 0.5")
ax.set_xlabel("Predicted confidence score")
ax.set_ylabel("Number of candidates")
ax.set_title("Confidence score distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 8 · Export of the final model

We retrain on **all data** (train + test) before saving.

In [ ]:
# Retrain on 100% of data
best_pipeline.fit(X, y)

# Save
joblib.dump(best_pipeline, MODEL_PATH)
print(f"✓ Model saved: {MODEL_PATH}")
print(f"  Model: {best_name}")
print(f"  Features ({len(FEATURE_COLUMNS)}) : {FEATURE_COLUMNS}")
print(f"  Classes: 0=Reject, 1=Invite")
print(f"  Trained on: {len(X)} rows")

# Quick sanity check — 15 values in FEATURE_COLUMNS order:
# total_years, num_pos, avg_tenure, edu_score, skills, has_cert, lang_count, section_score,
# max_lang_score, has_senior, gap_months, latest_dur, has_summary, num_certs, parse_quality
test_input = np.array([[6.0, 3, 24.0, 4, 12, 1, 3, 6, 5, 1, 0, 18, 1, 2, 2]])
loaded = joblib.load(MODEL_PATH)
pred = loaded.predict(test_input)[0]
proba = loaded.predict_proba(test_input)[0].max()
print(f"\nQuick test → {'Invite' if pred == 1 else 'Reject'} (confidence={proba:.2f})")

## 9 · Retrain on Real Data

Once you have processed CVs in the app:

```bash
# Export candidates
curl http://localhost:8000/api/v1/candidates/export.csv -o ml/candidates_export.csv
```

Then open the CSV, verify/correct the `recommendation` column (Invite or Reject), change `DATA_PATH` at the top of the notebook, and re-run all cells.

The new `model.joblib` is automatically loaded by the backend on the next prediction call — **no need to restart Docker**.